In [2]:
# importing packages

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import glob
from pathlib import Path

# Set up plotting style
plt.style.use('default')
sns.set_palette("husl")

# Try importing specdal for binary ASD reading
try:
    from specdal import Collection
    HAS_SPECDAL = True
except ImportError:
    HAS_SPECDAL = False

In [ ]:
# 1. ASD Parsing & Folder Traversal

def read_single_asd(filepath):
    """
    Reads a single .asd file into a pandas Series (index: wavelength, value: DN). 
    Uses specdal if installed; otherwise falls back to ASCII text parse.
    """
    if HAS_SPECDAL:
        c = Collection(name="single")
        c.read(filepath)
        spec = c.spectra[0]
        # Ensure wavelength is rounded/float index
        s = spec.measurement
        s.name = os.path.splittext(os.path.basename(filepath))[0]
        return s
    else:
        # Fallback if files were exported as ASCII/CSV ASD files
        df = pd.read_csv(filepath, sep=r'\s+|,', engine='python', comment='#', header=None)
        s = pd.Series(df.iloc[:, 1].values, index=df.iloc[:, 0].values)
        s.name = os.path.splitext(os.path.basename(filepath))[0]
        return s

def build_spectral_datasets_from_tree(root_dir):
    """Walks specimen folders -> condition folders (WR, freshwater, nowater, saltwater)
    -> loads .asd files, computes reflectance (Sample / WR), and returns DataFrames.
    """
    conditions = ["freshwater", "nowater", "saltwater"]
    raw_dict = {}
    corrected_dict = {}
    baseline_dict = {}
    metadata_records = []

    # Get all 25 specimen directories
    specimen_dirs = [
        d for d in glob.glob(os.path.join(root_dir, ))
    ]

In [5]:
import os
import glob
import struct
import numpy as np
import pandas as pd

# Check for specdal as an option
try:
    from specdal import Collection
    HAS_SPECDAL = True
except ImportError:
    HAS_SPECDAL = False


def read_binary_asd(filepath):
    """
    Direct binary reader for ASD FieldSpec v7/v8 files using Python's struct.
    Extracts starting wavelength, step, channel count, and raw float spectrum.
    """
    with open(filepath, "rb") as f:
        data = f.read()

    # ASD Header structure (standard ASD file format specs):
    # Offset 178: wavelen_init (float32)
    # Offset 182: wavelength_step (float32)
    # Offset 186: data_format (uint8) (0: float, 1: int32, 2: int16)
    # Offset 198: channels (uint16)
    # Offset 484: Start of spectra data stream
    try:
        wavelen_init = struct.unpack("<f", data[178:182])[0]
        wavelen_step = struct.unpack("<f", data[182:186])[0]
        data_format = struct.unpack("<B", data[186:187])[0]
        channels = struct.unpack("<H", data[198:200])[0]

        # Calculate wavelengths
        wavelengths = [round(wavelen_init + i * wavelen_step, 1) for i in range(channels)]

        # Read spectrum based on ASD data_format byte
        offset = 484
        if data_format == 0:  # 32-bit float (numeric raw DN / radiance)
            values = struct.unpack(f"<{channels}f", data[offset : offset + channels * 4])
        elif data_format == 1:  # 32-bit integer
            values = struct.unpack(f"<{channels}i", data[offset : offset + channels * 4])
        elif data_format == 2:  # 16-bit integer
            values = struct.unpack(f"<{channels}h", data[offset : offset + channels * 2])
        elif data_format == 3:  # 64-bit double
            values = struct.unpack(f"<{channels}d", data[offset : offset + channels * 8])
        else:
            # Default fallback for most ASD FieldSpec files
            values = struct.unpack(f"<{channels}f", data[offset : offset + channels * 4])

        s = pd.Series(values, index=wavelengths)
        s.name = os.path.splitext(os.path.basename(filepath))[0]
        return s

    except Exception:
        # If byte offsets vary by firmware version, fall back to specdal if installed
        if HAS_SPECDAL:
            c = Collection(name="single")
            c.read(filepath)
            spec = c.spectra[0]
            s = spec.measurement
            s.name = os.path.splitext(os.path.basename(filepath))[0]
            return s
        else:
            raise


def export_reflectance_to_csv(root_dir, output_filename="reflectance_matrix.csv"):
    """
    Traverses the specimen folders, loads .asd binary files, computes reflectance
    (Treatment / Specimen WR), and writes reflectance_matrix.csv and metadata_key.csv.
    """
    conditions = ["freshwater", "nowater", "saltwater"]
    reflectance_dict = {}
    metadata_records = []

    # Find specimen directories
    specimen_dirs = [
        d for d in glob.glob(os.path.join(root_dir, "*"))
        if os.path.isdir(d) and not os.path.basename(d).startswith('.') and os.path.basename(d) != 'plots'
    ]

    print(f"Found {len(specimen_dirs)} specimen directories in: {root_dir}")

    for spec_path in specimen_dirs:
        specimen_id = os.path.basename(spec_path)

        # 1. Load and average White Reference (WR) scans
        wr_dir = os.path.join(spec_path, "WR")
        wr_files = glob.glob(os.path.join(wr_dir, "*.asd")) + glob.glob(os.path.join(wr_dir, "*.ASD"))

        if not wr_files:
            print(f"⚠️ Warning: No .asd files found in {wr_dir}. Skipping {specimen_id}.")
            continue

        wr_series_list = []
        for wf in wr_files:
            try:
                wr_series_list.append(read_binary_asd(wf))
            except Exception as e:
                print(f"Error reading {wf}: {e}")

        if not wr_series_list:
            continue

        wr_df = pd.concat(wr_series_list, axis=1)
        mean_wr_spectrum = wr_df.mean(axis=1)

        # 2. Process each condition
        for cond in conditions:
            cond_dir = os.path.join(spec_path, cond)
            if not os.path.exists(cond_dir):
                continue

            sample_files = glob.glob(os.path.join(cond_dir, "*.asd")) + glob.glob(os.path.join(cond_dir, "*.ASD"))

            for sf in sample_files:
                sample_file_name = os.path.splitext(os.path.basename(sf))[0]
                col_name = f"{specimen_id}_{cond}_{sample_file_name}"

                try:
                    raw_s = read_binary_asd(sf)

                    # Reflectance = Sample DN / Baseline Mean WR DN
                    reflectance_s = raw_s / mean_wr_spectrum
                    reflectance_dict[col_name] = reflectance_s

                    metadata_records.append({
                        "sample_column": col_name,
                        "specimen": specimen_id,
                        "treatment": cond,
                        "original_file": os.path.basename(sf)
                    })
                except Exception as e:
                    print(f"Error reading {sf}: {e}")

    if not reflectance_dict:
        print("❌ No spectra were parsed. Check the file paths or permissions.")
        return None, None

    # Build final DataFrames
    reflectance_df = pd.DataFrame(reflectance_dict)
    reflectance_df.sort_index(inplace=True)
    reflectance_df.index.name = "wavelength_nm"

    # Export reflectance CSV
    output_path = os.path.join(root_dir, output_filename)
    reflectance_df.to_csv(output_path)

    # Export metadata key
    metadata_df = pd.DataFrame(metadata_records)
    metadata_path = os.path.join(root_dir, "metadata_key.csv")
    metadata_df.to_csv(metadata_path, index=False)

    print(f"\nProcessing complete:")
    print(f"  Reflectance matrix saved to: {output_path}")
    print(f"  Metadata key saved to:       {metadata_path}")
    print(f"  Shape: {reflectance_df.shape[0]} wavelengths × {reflectance_df.shape[1]} samples")

    return reflectance_df, metadata_df

In [ ]:
# Pass your root folder path (use forward slashes or raw string r"..." to avoid path escape errors)
root_data_dir = r"C:/Users/kimmi/OneDrive - Chapman University/SEACR/mkim_seacr_algaedata/asd_data/05.17.25_cabrillo.v2/05.17.25_cabrillo"

reflectance_df, metadata_df = export_reflectance_to_csv(root_data_dir)

Found 33 specimen directories in: C:\Users\kimmi\OneDrive - Chapman University\SEACR\mkim_seacr_algaedata\asd_data\05.17.25_cabrillo.v2\05.17.25_cabrillo
⚠️ Warning: No .asd files found in C:\Users\kimmi\OneDrive - Chapman University\SEACR\mkim_seacr_algaedata\asd_data\05.17.25_cabrillo.v2\05.17.25_cabrillo\algae_12\WR. Skipping algae_12.
⚠️ Warning: No .asd files found in C:\Users\kimmi\OneDrive - Chapman University\SEACR\mkim_seacr_algaedata\asd_data\05.17.25_cabrillo.v2\05.17.25_cabrillo\algae_18\WR. Skipping algae_18.

Processing complete:
  Reflectance matrix saved to: C:\Users\kimmi\OneDrive - Chapman University\SEACR\mkim_seacr_algaedata\asd_data\05.17.25_cabrillo.v2\05.17.25_cabrillo\reflectance_matrix.csv
  Metadata key saved to:       C:\Users\kimmi\OneDrive - Chapman University\SEACR\mkim_seacr_algaedata\asd_data\05.17.25_cabrillo.v2\05.17.25_cabrillo\metadata_key.csv
  Shape: 17795 wavelengths × 451 samples


In [9]:
import os
import glob
import struct
import numpy as np
import pandas as pd

# Check for specdal as an alternative reader
try:
    from specdal import Collection
    HAS_SPECDAL = True
except ImportError:
    HAS_SPECDAL = False


def read_binary_asd(filepath):
    """
    Direct binary reader for ASD FieldSpec v7/v8 files using Python's struct.
    Extracts starting wavelength, step, channel count, and raw float spectrum.
    """
    with open(filepath, "rb") as f:
        data = f.read()

    try:
        wavelen_init = struct.unpack("<f", data[178:182])[0]
        wavelen_step = struct.unpack("<f", data[182:186])[0]
        data_format = struct.unpack("<B", data[186:187])[0]
        channels = struct.unpack("<H", data[198:200])[0]

        wavelengths = [round(wavelen_init + i * wavelen_step, 1) for i in range(channels)]
        offset = 484

        if data_format == 0:
            values = struct.unpack(f"<{channels}f", data[offset : offset + channels * 4])
        elif data_format == 1:
            values = struct.unpack(f"<{channels}i", data[offset : offset + channels * 4])
        elif data_format == 2:
            values = struct.unpack(f"<{channels}h", data[offset : offset + channels * 2])
        elif data_format == 3:
            values = struct.unpack(f"<{channels}d", data[offset : offset + channels * 8])
        else:
            values = struct.unpack(f"<{channels}f", data[offset : offset + channels * 4])

        s = pd.Series(values, index=wavelengths)
        s.name = os.path.splitext(os.path.basename(filepath))[0]
        return s

    except Exception:
        if HAS_SPECDAL:
            c = Collection(name="single")
            c.read(filepath)
            spec = c.spectra[0]
            s = spec.measurement
            s.name = os.path.splitext(os.path.basename(filepath))[0]
            return s
        else:
            raise


def export_tidy_reflectance_to_csv(
    root_dir, 
    output_filename="tidy_reflectance_400_1000nm.csv",
    wl_min=400.0,
    wl_max=1000.0
):
    """
    Traverses specimen folders, pairs each treatment with its specimen WR,
    calculates reflectance, slices wavelengths between wl_min and wl_max,
    and outputs a long/tidy DataFrame.
    """
    conditions = ["freshwater", "nowater", "saltwater"]
    records = []

    # Locate specimen directories
    specimen_dirs = [
        d for d in glob.glob(os.path.join(root_dir, "*"))
        if os.path.isdir(d) and not os.path.basename(d).startswith('.') and os.path.basename(d) != 'plots'
    ]

    print(f"Found {len(specimen_dirs)} specimen directories in: {root_dir}")
    print(f"Filtering spectral range: {wl_min} nm – {wl_max} nm")

    for spec_path in specimen_dirs:
        specimen_id = os.path.basename(spec_path)

        # 1. Load and average White Reference (WR) scans for this specimen
        wr_dir = os.path.join(spec_path, "WR")
        wr_files = glob.glob(os.path.join(wr_dir, "*.asd")) + glob.glob(os.path.join(wr_dir, "*.ASD"))

        if not wr_files:
            print(f"⚠️ Warning: No .asd files found in {wr_dir}. Skipping {specimen_id}.")
            continue

        wr_series_list = []
        for wf in wr_files:
            try:
                wr_series_list.append(read_binary_asd(wf))
            except Exception as e:
                print(f"Error reading {wf}: {e}")

        if not wr_series_list:
            continue

        wr_df = pd.concat(wr_series_list, axis=1)
        mean_wr_spectrum = wr_df.mean(axis=1)

        # 2. Process each condition
        for cond in conditions:
            cond_dir = os.path.join(spec_path, cond)
            if not os.path.exists(cond_dir):
                continue

            sample_files = glob.glob(os.path.join(cond_dir, "*.asd")) + glob.glob(os.path.join(cond_dir, "*.ASD"))
            sample_files.sort()

            for rep_idx, sf in enumerate(sample_files, start=1):
                try:
                    raw_s = read_binary_asd(sf)

                    # Reflectance = Sample DN / Baseline Mean WR DN
                    reflectance_s = (raw_s / mean_wr_spectrum).dropna()

                    # Filter exclusively to 400 - 1000 nm
                    reflectance_s = reflectance_s.loc[
                        (reflectance_s.index >= wl_min) & (reflectance_s.index <= wl_max)
                    ]

                    # Convert the spectrum series into long-format rows
                    temp_df = pd.DataFrame({
                        "wavelength_nm": reflectance_s.index,
                        "specimen": specimen_id,
                        "species": "",  # Blank column for manual labeling
                        "treatment": cond,
                        "replicate": rep_idx,
                        "reflectance": reflectance_s.values,
                        "original_file": os.path.basename(sf)
                    })

                    records.append(temp_df)

                except Exception as e:
                    print(f"Error reading {sf}: {e}")

    if not records:
        print("❌ No spectra were parsed.")
        return None

    # Concatenate all into a single long dataframe
    tidy_df = pd.concat(records, ignore_index=True)

    # Sort rows cleanly
    tidy_df.sort_values(by=["specimen", "treatment", "replicate", "wavelength_nm"], inplace=True)

    # Save to CSV
    output_path = os.path.join(root_dir, output_filename)
    tidy_df.to_csv(output_path, index=False)

    print(f"\nProcessing complete:")
    print(f"  Tidy file saved to: {output_path}")
    print(f"  Total data rows:    {len(tidy_df):,}")
    print(f"  Wavelength bands:   {tidy_df['wavelength_nm'].nunique()} ({tidy_df['wavelength_nm'].min()} - {tidy_df['wavelength_nm'].max()} nm)")
    print(f"  Specimens included: {tidy_df['specimen'].nunique()}")

    return tidy_df

In [10]:
root_data_dir = r"C:\Users\kimmi\OneDrive - Chapman University\SEACR\mkim_seacr_algaedata\asd_data\05.17.25_cabrillo.v2\05.17.25_cabrillo"

tidy_df = export_tidy_reflectance_to_csv(root_data_dir, wl_min=400.0, wl_max=1000.0)

Found 33 specimen directories in: C:\Users\kimmi\OneDrive - Chapman University\SEACR\mkim_seacr_algaedata\asd_data\05.17.25_cabrillo.v2\05.17.25_cabrillo
Filtering spectral range: 400.0 nm – 1000.0 nm
⚠️ Warning: No .asd files found in C:\Users\kimmi\OneDrive - Chapman University\SEACR\mkim_seacr_algaedata\asd_data\05.17.25_cabrillo.v2\05.17.25_cabrillo\algae_12\WR. Skipping algae_12.
⚠️ Warning: No .asd files found in C:\Users\kimmi\OneDrive - Chapman University\SEACR\mkim_seacr_algaedata\asd_data\05.17.25_cabrillo.v2\05.17.25_cabrillo\algae_18\WR. Skipping algae_18.

Processing complete:
  Tidy file saved to: C:\Users\kimmi\OneDrive - Chapman University\SEACR\mkim_seacr_algaedata\asd_data\05.17.25_cabrillo.v2\05.17.25_cabrillo\tidy_reflectance_400_1000nm.csv
  Total data rows:    0
  Wavelength bands:   0 (nan - nan nm)
  Specimens included: 0


In [21]:
# ignoring hermit snail, only algae specimen
# tidy, long format of data
# only looking at reflectances between 400 nm and 1000 nm

import os
import glob
import re
import struct
import numpy as np
import pandas as pd


# 1. Exact Binary ASD Reader (Verified via Test A)
def read_as7_binary(filepath):
    """
    Decodes ASD FieldSpec Version 7 (as7) binary files.
    Reads 2151 channels as 64-bit doubles starting at offset 484.
    """
    with open(filepath, "rb") as f:
        data = f.read()

    channels = 2151
    # Physical standard wavelength array: 350 to 2500 nm (integer bands)
    wavelengths = np.arange(350, 350 + channels, dtype=int)

    # Offset 484, 64-bit doubles (<d)
    offset = 484
    values = struct.unpack(f"<{channels}d", data[offset : offset + channels * 8])

    s = pd.Series(values, index=wavelengths, dtype=float)
    s.name = os.path.splitext(os.path.basename(filepath))[0]
    return s


# 2. Main Traversal and Tidy Exporter
def export_tidy_reflectance_to_csv(
    root_dir, 
    output_filename="tidy_reflectance_algae_400_1000nm.csv",
    wl_min=400,
    wl_max=1000
):
    target_conditions = ["freshwater", "nowater", "saltwater"]
    records = []

    # Filter strictly for folders starting with "algae_"
    all_subdirs = [
        d for d in os.listdir(root_dir)
        if os.path.isdir(os.path.join(root_dir, d)) and d.lower().startswith("algae_")
    ]

    def get_algae_num(name):
        match = re.search(r'\d+', name)
        return int(match.group()) if match else 9999

    specimen_folder_names = sorted(all_subdirs, key=get_algae_num)
    specimen_dirs = [os.path.join(root_dir, d) for d in specimen_folder_names]

    print(f"Targeting {len(specimen_dirs)} algae specimen folders:")
    print(f"{specimen_folder_names}\n")
    print(f"Slicing spectral bands: {wl_min} nm – {wl_max} nm\n")

    for spec_path in specimen_dirs:
        specimen_id = os.path.basename(spec_path)
        subdirs = {d.lower().strip(): d for d in os.listdir(spec_path) if os.path.isdir(os.path.join(spec_path, d))}

        # Match WR folder
        wr_folder_name = subdirs.get("wr")
        if not wr_folder_name:
            print(f"⚠️ Skipping '{specimen_id}': 'WR' folder not found.")
            continue

        wr_dir = os.path.join(spec_path, wr_folder_name)
        wr_files = sorted(glob.glob(os.path.join(wr_dir, "*.[aA][sS][dD]")))

        if not wr_files:
            print(f"⚠️ Skipping '{specimen_id}': No .asd files in {wr_dir}")
            continue

        # Load and average White Reference scans
        wr_series_list = []
        for wf in wr_files:
            try:
                wr_series_list.append(read_as7_binary(wf))
            except Exception as e:
                print(f"Error reading WR file {os.path.basename(wf)}: {e}")

        if not wr_series_list:
            continue

        mean_wr = pd.concat(wr_series_list, axis=1).mean(axis=1)

        specimen_sample_count = 0
        for cond_key in target_conditions:
            cond_folder_name = subdirs.get(cond_key)
            if not cond_folder_name:
                continue

            cond_dir = os.path.join(spec_path, cond_folder_name)
            sample_files = sorted(glob.glob(os.path.join(cond_dir, "*.[aA][sS][dD]")))

            for rep_idx, sf in enumerate(sample_files, start=1):
                try:
                    raw_s = read_as7_binary(sf)

                    # Compute reflectance: Sample DN / Specimen Mean WR DN
                    refl_values = raw_s.values / mean_wr.values
                    wl_array = np.array(raw_s.index)

                    # Filter strictly to 400 - 1000 nm
                    mask = (wl_array >= wl_min) & (wl_array <= wl_max)
                    filtered_wl = wl_array[mask]
                    filtered_refl = refl_values[mask]

                    temp_df = pd.DataFrame({
                        "wavelength_nm": filtered_wl,
                        "specimen": specimen_id,
                        "specimen_num": get_algae_num(specimen_id),
                        "species": "",  # Blank column ready for your manual labeling
                        "treatment": cond_key,
                        "replicate": rep_idx,
                        "reflectance": filtered_refl,
                        "original_file": os.path.basename(sf)
                    })

                    records.append(temp_df)
                    specimen_sample_count += 1

                except Exception as e:
                    print(f"Error reading {os.path.basename(sf)}: {e}")

        print(f"✓ {specimen_id}: Processed {specimen_sample_count} scans.")

    if not records:
        print("\n❌ No records were created.")
        return None

    # Merge all into one tidy DataFrame
    tidy_df = pd.concat(records, ignore_index=True)

    # Sort cleanly by specimen, treatment, replicate, and wavelength
    tidy_df.sort_values(by=["specimen_num", "treatment", "replicate", "wavelength_nm"], inplace=True)
    tidy_df.drop(columns=["specimen_num"], inplace=True)

    # Save to CSV
    output_path = os.path.join(root_dir, output_filename)
    tidy_df.to_csv(output_path, index=False)

    print(f"\n==========================================")
    print(f"✅ Successfully exported!")
    print(f"Saved to:           {output_path}")
    print(f"Total rows:         {len(tidy_df):,}")
    print(f"Algae specimens:    {tidy_df['specimen'].nunique()}")
    print(f"Wavelength bands:   {tidy_df['wavelength_nm'].min()} - {tidy_df['wavelength_nm'].max()} nm ({tidy_df['wavelength_nm'].nunique()} bands)")
    print(f"Treatments present: {tidy_df['treatment'].unique().tolist()}")
    print(f"==========================================")

    return tidy_df


# 3. Execution
root_data_dir = r"C:\Users\kimmi\OneDrive - Chapman University\SEACR\mkim_seacr_algaedata\asd_data\mkim_251121\mkim_251121"

tidy_df = export_tidy_reflectance_to_csv(root_data_dir, wl_min=400, wl_max=1000)

Targeting 24 algae specimen folders:
['algae_1', 'algae_2', 'algae_3', 'algae_4', 'algae_5', 'algae_6', 'algae_7', 'algae_8', 'algae_9', 'algae_10', 'algae_11', 'algae_12', 'algae_13', 'algae_14', 'algae_15', 'algae_16', 'algae_17', 'algae_18', 'algae_19', 'algae_20', 'algae_21', 'algae_22', 'algae_23', 'algae_24']

Slicing spectral bands: 400 nm – 1000 nm

✓ algae_1: Processed 15 scans.
✓ algae_2: Processed 15 scans.
✓ algae_3: Processed 15 scans.
✓ algae_4: Processed 15 scans.
✓ algae_5: Processed 15 scans.
✓ algae_6: Processed 15 scans.
✓ algae_7: Processed 15 scans.
✓ algae_8: Processed 15 scans.
✓ algae_9: Processed 15 scans.
✓ algae_10: Processed 15 scans.
✓ algae_11: Processed 15 scans.
✓ algae_12: Processed 15 scans.
✓ algae_13: Processed 15 scans.
✓ algae_14: Processed 15 scans.
✓ algae_15: Processed 15 scans.
✓ algae_16: Processed 15 scans.
✓ algae_17: Processed 15 scans.
✓ algae_18: Processed 15 scans.
✓ algae_19: Processed 15 scans.
✓ algae_20: Processed 15 scans.
✓ algae_2

In [17]:
test_file = r"C:\Users\kimmi\OneDrive - Chapman University\SEACR\mkim_seacr_algaedata\asd_data\05.17.25_cabrillo.v2\05.17.25_cabrillo\algae_1\freshwater\cabrillo_5.17.25_00038.asd"
# (Or any valid .asd path from algae_1)

s = read_as7_binary(test_file)
print("Channels count:", len(s))
print("First 5 index values (wavelengths):", list(s.index[:5]))
print("Last 5 index values (wavelengths):", list(s.index[-5:]))
print("First 5 reflectance/DN values:", list(s.values[:5]))

Channels count: 2151
First 5 index values (wavelengths): [0, 3188350521909894942031872, 6376701043819789884063744, 9565051565729684826095616, 12753402087639579768127488]
Last 5 index values (wavelengths): [6842200220018634545600397312, 6845388570540544440542429184, 6848576921062454335484461056, 6851765271584364230426492928, 6854953622106274125368524800]
First 5 reflectance/DN values: [np.float64(3.318133731749717e-32), np.float64(2.606830596923828), np.float64(-4.163672183816643e-27), np.float64(2.604264736175537), np.float64(-1.662016297433953e-27)]


In [19]:
import struct
import numpy as np

test_file = r"C:\Users\kimmi\OneDrive - Chapman University\SEACR\mkim_seacr_algaedata\asd_data\05.17.25_cabrillo.v2\05.17.25_cabrillo\algae_1\freshwater\cabrillo_5.17.25_00038.asd"

with open(test_file, 'rb') as f:
    data = f.read()

channels = 2151
print(f"Total file size: {len(data)} bytes")

# Check 1: 64-bit doubles (8 bytes per channel = 17,208 bytes)
if len(data) >= channels * 8:
    vals_double = struct.unpack(f"<{channels}d", data[484 : 484 + channels * 8])
    print("\n--- Test A: 64-bit Doubles (Offset 484) ---")
    print("Values 100 to 105 (around 450nm):", [round(v, 4) for v in vals_double[100:105]])

# Check 2: 64-bit doubles anchored from the END of file (bypasses comment header)
if len(data) >= channels * 8:
    vals_double_end = struct.unpack(f"<{channels}d", data[-channels * 8 :])
    print("\n--- Test B: 64-bit Doubles (End of File) ---")
    print("Values 100 to 105 (around 450nm):", [round(v, 4) for v in vals_double_end[100:105]])

# Check 3: 32-bit floats anchored from the END of file
vals_float_end = struct.unpack(f"<{channels}f", data[-channels * 4 :])
print("\n--- Test C: 32-bit Floats (End of File) ---")
print("Values 100 to 105 (around 450nm):", [round(v, 4) for v in vals_float_end[100:105]])

Total file size: 34997 bytes

--- Test A: 64-bit Doubles (Offset 484) ---
Values 100 to 105 (around 450nm): [76.4346, 78.3389, 79.636, 83.6934, 87.0473]

--- Test B: 64-bit Doubles (End of File) ---
Values 100 to 105 (around 450nm): [8.626840722371784e+192, -4.7763613916666806e+272, 1.8094160880845832e+183, 0.0, 0.0]

--- Test C: 32-bit Floats (End of File) ---
Values 100 to 105 (around 450nm): [-0.0, 0.0001, -0.0, -9.779981326002597e+33, -1973438080.0]
